In [2]:
import os, json, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# === 1) Inputs (main split predictions you already saved) ===
RESNET = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/Resnet-18 Model/predictions/resnet18_val_test_predictions.csv"
VIT    = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/ViT-Model/predictions/vit_lora_val_test_predictions.csv"
EFFNET = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/predictions/efficientnet_b0_val_test_predictions.csv"

OUTDIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble"
os.makedirs(f"{OUTDIR}/predictions", exist_ok=True)
os.makedirs(f"{OUTDIR}/reports", exist_ok=True)

# === 2) Load & align by filepath + split ===
def load_preds(path, name):
    df = pd.read_csv(path)
    # expected cols: filepath, split, y_true, prob_cancer
    need = {"filepath","split","y_true","prob_cancer"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    df = df[["filepath","split","y_true","prob_cancer"]].copy()
    df = df.rename(columns={"prob_cancer": f"prob_{name}"})
    return df

r = load_preds(RESNET, "resnet")
v = load_preds(VIT,    "vit")
e = load_preds(EFFNET, "effnet")

df = r.merge(v, on=["filepath","split","y_true"], how="inner").merge(e, on=["filepath","split","y_true"], how="inner")
assert len(df)>0, "No overlapping rows across models."

# === 3) Simple soft-vote (equal weights). If you want weights, adjust here. ===
df["prob_ens"] = (df["prob_resnet"] + df["prob_vit"] + df["prob_effnet"]) / 3.0

# === 4) Pick τ* on VAL (either max-F1 or precision floor) ===
PREC_TARGET = 0.98   # set to None for max-F1; keep 0.98 to mimic your policy

def pick_tau_on_val(y_true, p_val, prec_target=PREC_TARGET):
    thr = np.unique(np.r_[0.0, p_val, 1.0])
    stats = []
    for t in thr:
        yhat = (p_val >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y_true, yhat, average="binary", zero_division=0)
        stats.append((t, pr, rc, f1))
    stats = np.array(stats, dtype=float)

    if prec_target is not None:
        cand = stats[stats[:,1] >= prec_target]
        if cand.size > 0:
            # prefer higher recall, tie-break to lower τ
            cand = cand[np.lexsort((cand[:,0], -cand[:,2]))]
            t = float(cand[0,0])
            # safety: ensure ≥1 positive
            if (p_val >= t).sum() == 0:
                ok = [row for row in stats if (p_val >= row[0]).sum() >= 1]
                ok.sort(key=lambda r: (r[3], -r[0]))  # best F1, then smaller τ
                t = float(ok[-1][0]) if ok else 0.0
            return t, f"prec>={prec_target:.2f}"
    # max-F1, tie -> higher recall then lower τ
    best_f1 = stats[:,3].max()
    tied = stats[np.abs(stats[:,3]-best_f1) <= 1e-12]
    tied = tied[np.lexsort((tied[:,0], -tied[:,2]))]
    return float(tied[0,0]), "maxF1"

def eval_split(d, t):
    y = d["y_true"].values.astype(int)
    p = d["prob_ens"].values
    yhat = (p >= t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    auc = roc_auc_score(y, p) if len(np.unique(y))>1 else float('nan')
    cm  = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    acc = (yhat==y).mean()
    return dict(acc=float(acc), prec=float(pr), rec=float(rc), f1=float(f1), auc=float(auc), cm=cm)

val  = df[df["split"]=="val"].copy()
test = df[df["split"]=="test"].copy()
assert len(val)>0 and len(test)>0, "VAL/TEST missing."

tau_star, policy = pick_tau_on_val(val["y_true"].values, val["prob_ens"].values)

val_metrics  = eval_split(val,  tau_star)
test_metrics = eval_split(test, tau_star)

print(f"Chosen τ*={tau_star:.6f}  policy={policy}")
print("[VAL @ τ*]", val_metrics)
print("[TEST @ τ*]", test_metrics)

# === 5) Save predictions + τ* ===
out_csv = f"{OUTDIR}/predictions/ensemble_val_test_predictions.csv"
out_tau = f"{OUTDIR}/reports/operating_threshold.txt"
df_out = df.copy()
df_out["prob_ens_tauStar"] = tau_star
df_out.to_csv(out_csv, index=False)
with open(out_tau, "w") as f:
    f.write(f"{tau_star:.6f}\n")

manifest = {
    "models": {
        "resnet_csv": RESNET,
        "vit_csv": VIT,
        "effnet_csv": EFFNET
    },
    "tau_star": float(tau_star),
    "policy": policy,
    "metrics": {"val": val_metrics, "test": test_metrics},
    "outputs": {"predictions_csv": out_csv, "tau_file": out_tau}
}
with open(f"{OUTDIR}/reports/manifest_ensemble.json","w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved:")
print(" -", out_csv)
print(" -", out_tau)
print(" -", f"{OUTDIR}/reports/manifest_ensemble.json")


Chosen τ*=0.920774  policy=prec>=0.98
[VAL @ τ*] {'acc': 1.0, 'prec': 1.0, 'rec': 1.0, 'f1': 1.0, 'auc': 1.0, 'cm': [[734, 0], [0, 6]]}
[TEST @ τ*] {'acc': 1.0, 'prec': 1.0, 'rec': 1.0, 'f1': 1.0, 'auc': 1.0, 'cm': [[367, 0], [0, 3]]}

Saved:
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble/predictions/ensemble_val_test_predictions.csv
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble/reports/operating_threshold.txt
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble/reports/manifest_ensemble.json


In [6]:
import os, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# paths
ENS = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble/predictions/ensemble_val_test_predictions.csv"
TAU = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Ensemble/reports/operating_threshold.txt"
META = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/master_metadata_realpaths.csv"

# load predictions + tau
df = pd.read_csv(ENS)
with open(TAU,"r") as f: tau = float(f.read().strip())

# merge BOTH source and label from metadata
meta = pd.read_csv(META)[["filepath","source","label"]]
df = df.merge(meta, on="filepath", how="left")

# derive label from y_true if missing/NA
if "label" not in df.columns or df["label"].isna().any():
    if "y_true" not in df.columns:
        raise ValueError("Neither label nor y_true present; cannot continue.")
    df["y_true"] = df["y_true"].astype(int)
    df["label"] = np.where(df["y_true"]==1, "cancer", "healthy")

# pick prob column
prob_col = "prob_ens" if "prob_ens" in df.columns else next((c for c in df.columns if c.startswith("prob")), None)
assert prob_col is not None, f"No probability column found in {df.columns.tolist()}"

def metrics_at_tau(d, t, name):
    y = (d["label"].astype(str).str.lower()=="cancer").astype(int).values
    p = d[prob_col].values
    yhat = (p>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y,yhat,average="binary",zero_division=0)
    auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
    cm  = confusion_matrix(y,yhat,labels=[0,1]).tolist()
    acc = (yhat==y).mean()
    print(f"[{name}] acc={acc:.3f}  prec={pr:.3f}  rec={rc:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)

kaggle_sources = {"kaggle_cancer_raw","kaggle_normal_raw"}

for split in ["val","test"]:
    d = df[df["split"]==split].copy()

    print("\n====", split.upper(), "====  (τ* =", f"{tau:.6f}", ")")
    # Kaggle-only
    kag = d[d["source"].isin(kaggle_sources)].copy()
    if len(kag):
        metrics_at_tau(kag, tau, f"{split} Kaggle-only")
        print("  breakdown:\n", kag["label"].value_counts())
    else:
        print(f"[{split} Kaggle-only] no rows")

    # No-Kaggle
    nok = d[~d["source"].isin(kaggle_sources)].copy()
    if len(nok):
        metrics_at_tau(nok, tau, f"{split} no-Kaggle")
        print("  sources:", nok["source"].value_counts().to_dict())
    else:
        print(f"[{split} no-Kaggle] no rows")



==== VAL ====  (τ* = 0.920774 )
[val Kaggle-only] acc=0.909  prec=1.000  rec=0.833  f1=0.909  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[5, 0], [1, 5]]
  breakdown:
 label
cancer     6
healthy    5
Name: count, dtype: int64
[val no-Kaggle] acc=1.000  prec=0.000  rec=0.000  f1=0.000  auc=nan
CM (rows=true [healthy,cancer], cols=pred):
 [[729, 0], [0, 0]]
  sources: {'Cough_Audio_Coswera': 562, 'Cough_Audio_COUGHVID': 167}

==== TEST ====  (τ* = 0.920774 )
[test Kaggle-only] acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[2, 0], [0, 3]]
  breakdown:
 label
cancer     3
healthy    2
Name: count, dtype: int64
[test no-Kaggle] acc=1.000  prec=0.000  rec=0.000  f1=0.000  auc=nan
CM (rows=true [healthy,cancer], cols=pred):
 [[365, 0], [0, 0]]
  sources: {'Cough_Audio_Coswera': 281, 'Cough_Audio_COUGHVID': 84}


In [8]:
# === RESNET18 LOSO TRAIN + PREDICT ===
import os, json, math, random, numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix
from torchvision import models, transforms as T

# ---- paths
BASE_LOSO = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
OUT_ROOT  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
MODEL_DIR = "resnet18"  # per-fold subdir under LOSO/<fold>/<MODEL_DIR>

# ---- helpers
def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class GrayTo3(torch.nn.Module):
    def forward(self, x):  # x: (1,H,W)
        return x.repeat(3,1,1)

def make_tfms(train=True):
    if train:
        return T.Compose([
            T.Grayscale(num_output_channels=1),
            T.Resize((224,224)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ToTensor(),
            GrayTo3(),
            T.Normalize(mean=[0.5,0.5,0.5], std=[0.25,0.25,0.25]),
        ])
    else:
        return T.Compose([
            T.Grayscale(num_output_channels=1),
            T.Resize((224,224)),
            T.ToTensor(),
            GrayTo3(),
            T.Normalize(mean=[0.5,0.5,0.5], std=[0.25,0.25,0.25]),
        ])

class ImgCSV(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        self.fp   = df["filepath"].tolist()
        self.y    = (df["label"].astype(str).str.lower()=="cancer").astype(int).values
        self.tfm  = tfm
    def __len__(self): return len(self.fp)
    def __getitem__(self, i):
        im = Image.open(self.fp[i]).convert("L")
        x  = self.tfm(im)
        return x, int(self.y[i])

def counts_from_labels(lbls):
    c0 = int((lbls==0).sum()); c1 = int((lbls==1).sum())
    return np.array([c0,c1], dtype=np.int64)

def choose_tau(val_probs, val_y, policy_prec=0.98):
    # grid search over thresholds
    taus = np.unique(np.clip(np.r_[0.0, val_probs, 1.0], 0,1))
    best_f1, best_tau = -1, 0.5
    pr_best = 0
    # best-F1
    for t in taus:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average='binary', zero_division=0)
        if f1>best_f1: best_f1, best_tau, pr_best = f1, float(t), pr
    # precision-targeted
    taus_desc = taus[::-1]
    for t in taus_desc:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average='binary', zero_division=0)
        if pr >= policy_prec:
            return float(t), "prec>=%.2f" % policy_prec
    return best_tau, "maxF1"

def train_one_fold(fold_dir, device="cuda", seed=42):
    seed_all(seed)
    fold_dir = Path(fold_dir)
    split_dir = fold_dir / "dedup_strict" if (fold_dir/"dedup_strict").exists() else fold_dir  # use dedup if present
    train_csv = split_dir / "train.csv"
    val_csv   = split_dir / "val.csv"
    test_csv  = split_dir / "test.csv"
    assert train_csv.exists() and val_csv.exists() and test_csv.exists(), f"Missing CSVs under {split_dir}"

    out_dir = fold_dir / MODEL_DIR
    (out_dir/"checkpoints").mkdir(parents=True, exist_ok=True)
    (out_dir/"predictions").mkdir(parents=True, exist_ok=True)
    (out_dir/"reports").mkdir(parents=True, exist_ok=True)

    # data
    train_ds = ImgCSV(str(train_csv), make_tfms(True))
    val_ds   = ImgCSV(str(val_csv),   make_tfms(False))
    test_ds  = ImgCSV(str(test_csv),  make_tfms(False))

    # sampler for imbalance
    y_tr = np.array([y for _,y in train_ds])
    cnts = counts_from_labels(y_tr)
    class_w = (cnts.sum() / (2.0*np.maximum(cnts,1))).astype(np.float32)
    sample_w = np.where(y_tr==0, class_w[0], class_w[1]).astype(np.float32)
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

    # model
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 2)
    m.to(device)

    crit = nn.CrossEntropyLoss(weight=torch.tensor(class_w, device=device))
    opt  = AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)

    best_auc, best_ep = -1, 0
    BEST = out_dir/"checkpoints"/"resnet18_best.pt"

    def run_epoch(loader, train=True):
        m.train(train)
        losses, ys, ps = [], [], []
        for xb,yb in loader:
            xb,yb = xb.to(device), yb.to(device)
            with torch.set_grad_enabled(train):
                logits = m(xb)
                loss = crit(logits, yb)
                if train:
                    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            probs = F.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            losses.append(loss.item()); ys.append(yb.cpu().numpy()); ps.append(probs)
        y = np.concatenate(ys); p = np.concatenate(ps)
        auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
        pr,rc,f1,_ = precision_recall_fscore_support(y,(p>=0.5).astype(int),average='binary',zero_division=0)
        return np.mean(losses), f1, auc, y, p

    # train
    patience, waited = 3, 0
    for ep in range(1, 30+1):
        tr = run_epoch(train_loader, True)
        vl = run_epoch(val_loader, False)
        print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[1]:.3f} auc {tr[2]:.3f} | val loss {vl[0]:.4f} f1 {vl[1]:.3f} auc {vl[2]:.3f}")
        if (vl[2] if not math.isnan(vl[2]) else -1) > best_auc:
            best_auc, best_ep, waited = vl[2], ep, 0
            torch.save(m.state_dict(), BEST)
        else:
            waited += 1
            if waited>=patience:
                print(f"Early stopping at {ep} (best val AUC at {best_ep}: {best_auc:.3f})")
                break

    # reload best
    m.load_state_dict(torch.load(BEST, map_location=device))
    m.eval()

    # collect val/test probs for τ* and report
    with torch.no_grad():
        vl = run_epoch(val_loader, False)  # (loss,f1,auc,y,p)
        ts = run_epoch(test_loader, False)

    tau, policy = choose_tau(vl[4], vl[3], policy_prec=0.98)
    (out_dir/"reports"/"operating_threshold.txt").write_text(f"{tau:.6f}")

    def at_tau(name, y, p, t):
        yhat = (p>=t).astype(int)
        pr,rc,f1,_ = precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
        cm  = confusion_matrix(y,yhat,labels=[0,1]).tolist()
        acc = (yhat==y).mean()
        print(f"[{name}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f} | τ*={t:.6f} ({policy})")
        return dict(acc=float(acc),prec=float(pr),rec=float(rc),f1=float(f1),auc=float(auc),cm=cm,tau=float(t))

    val_summary  = at_tau("VAL @τ*",  vl[3], vl[4], tau)
    test_summary = at_tau("TEST @τ*", ts[3], ts[4], tau)

    # save predictions CSV (val+test)
    def dump_preds(loader, split):
        rows=[]
        with torch.no_grad():
            for xb,yb in loader:
                logits = m(xb.to(device))
                p = F.softmax(logits,dim=1)[:,1].cpu().numpy()
                for i,(xx,yy) in enumerate(zip(yb.numpy().tolist(), p.tolist())):
                    rows.append(dict(filepath=loader.dataset.fp[i], y_true=int(yy is not None and loader.dataset.y[i]), prob_resnet=float(p[i]), split=split))
        return rows

    # build rows using dataset indices (safer)
    def rows_from_ds(ds, probs, split):
        return [dict(filepath=ds.fp[i], y_true=int(ds.y[i]), prob_resnet=float(probs[i]), split=split) for i in range(len(ds))]

    val_rows  = rows_from_ds(val_ds,  vl[4],  "val")
    test_rows = rows_from_ds(test_ds, ts[4],  "test")
    pred_csv  = out_dir/"predictions"/"resnet18_val_test_predictions.csv"
    pd.DataFrame(val_rows+test_rows).to_csv(pred_csv, index=False)

    manifest = {
      "model":"ResNet-18",
      "tau_star": float(tau),
      "paths": {"checkpoint": str(BEST), "predictions_csv": str(pred_csv), "tau_file": str(out_dir/'reports'/'operating_threshold.txt')},
      "val_at_tau": val_summary, "test_at_tau": test_summary
    }
    (out_dir/"reports"/"manifest_resnet18.json").write_text(json.dumps(manifest, indent=2))
    return str(pred_csv), float(tau)

# ---- run for all LOSO folds present
folds = [p for p in Path(BASE_LOSO).iterdir() if p.is_dir()]
print("Folds:", [f.name for f in folds])
for f in folds:
    print("\n=== RESNET18:", f.name, "===")
    _ = train_one_fold(str(f))


Folds: ['Cough_Audio_Coswera', 'Cough_Audio_COUGHVID']

=== RESNET18: Cough_Audio_Coswera ===
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 54.3MB/s]


Epoch 01 | train loss 0.0290 f1 0.938 auc 0.996 | val loss 0.2801 f1 0.400 auc 1.000
Epoch 02 | train loss 0.0015 f1 0.994 auc 1.000 | val loss 0.0155 f1 1.000 auc 1.000
Epoch 03 | train loss 0.0003 f1 1.000 auc 1.000 | val loss 0.0159 f1 1.000 auc 1.000
Epoch 04 | train loss 0.0002 f1 1.000 auc 1.000 | val loss 0.0186 f1 1.000 auc 1.000
Early stopping at 4 (best val AUC at 1: 1.000)
[VAL @τ*] acc=0.965 prec=1.000 rec=0.250 f1=0.400 auc=1.000 | τ*=0.999926 (prec>=0.98)
[TEST @τ*] acc=0.989 prec=0.000 rec=0.000 f1=0.000 auc=1.000 | τ*=0.999926 (prec>=0.98)

=== RESNET18: Cough_Audio_COUGHVID ===
Epoch 01 | train loss 0.0167 f1 0.986 auc 0.997 | val loss 0.0163 f1 0.706 auc 1.000
Epoch 02 | train loss 0.0006 f1 0.995 auc 1.000 | val loss 0.0109 f1 0.706 auc 1.000
Epoch 03 | train loss 0.0005 f1 0.997 auc 1.000 | val loss 0.0079 f1 0.800 auc 1.000
Epoch 04 | train loss 0.0002 f1 0.999 auc 1.000 | val loss 0.0039 f1 0.800 auc 1.000
Epoch 05 | train loss 0.0002 f1 0.998 auc 1.000 | val loss

In [10]:
# === VIT-MLP-LoRA LOSO TRAIN + PREDICT ===
import os, json, math, random, numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix
from torchvision import models, transforms as T

BASE_LOSO = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
OUT_ROOT  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
MODEL_DIR = "vit_lora"

class GrayTo3(torch.nn.Module):
    def forward(self, x): return x.repeat(3,1,1)

def make_tfms(train=True):
    if train:
        return T.Compose([
            T.Grayscale(num_output_channels=1),
            T.Resize((224,224)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ToTensor(), GrayTo3(),
            T.Normalize([0.5]*3,[0.25]*3),
        ])
    else:
        return T.Compose([
            T.Grayscale(num_output_channels=1),
            T.Resize((224,224)),
            T.ToTensor(), GrayTo3(),
            T.Normalize([0.5]*3,[0.25]*3),
        ])

class ImgCSV(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        self.fp = df["filepath"].tolist()
        self.y  = (df["label"].astype(str).str.lower()=="cancer").astype(int).values
        self.tfm = tfm
    def __len__(self): return len(self.fp)
    def __getitem__(self, i):
        x = Image.open(self.fp[i]).convert("L")
        return self.tfm(x), int(self.y[i])

# simple LoRA module for MLP linear layers
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=8, dropout=0.0):
        super().__init__()
        self.base = base
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.A = nn.Linear(base.in_features, r, bias=False)
        self.B = nn.Linear(r, base.out_features, bias=False)
        self.drop = nn.Dropout(dropout)
        # freeze base
        for p in self.base.parameters(): p.requires_grad=False
    def forward(self, x):
        return self.base(x) + self.drop(self.B(self.A(x))) * self.scaling

def inject_mlp_lora(vit, r=8, alpha=8, dropout=0.0):
    # torchvision ViT: encoder.layers -> EncoderBlock -> mlp: Sequential(Linear, GELU, Dropout, Linear, Dropout)
    cnt=0
    for blk in vit.encoder.layers:
        lin1 = blk.mlp[0]; lin2 = blk.mlp[3]
        blk.mlp[0] = LoRALinear(lin1, r=r, alpha=alpha, dropout=dropout)
        blk.mlp[3] = LoRALinear(lin2, r=r, alpha=alpha, dropout=dropout)
        cnt+=2
    return cnt

def choose_tau(val_probs, val_y, policy_prec=0.98):
    taus = np.unique(np.clip(np.r_[0.0, val_probs, 1.0], 0,1))
    # best-F1
    best_f1, best_tau = -1, 0.5
    for t in taus:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y,yhat,average='binary',zero_division=0)
        if f1>best_f1: best_f1, best_tau = f1, float(t)
    # precision-target
    for t in taus[::-1]:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y,yhat,average='binary',zero_division=0)
        if pr>=policy_prec: return float(t), "prec>=%.2f"%policy_prec
    return best_tau, "maxF1"

def train_one_fold(fold_dir, device="cuda", seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    fold_dir = Path(fold_dir)
    split_dir = fold_dir / "dedup_strict" if (fold_dir/"dedup_strict").exists() else fold_dir
    train_csv = split_dir / "train.csv"
    val_csv   = split_dir / "val.csv"
    test_csv  = split_dir / "test.csv"
    out_dir   = fold_dir / MODEL_DIR
    (out_dir/"checkpoints").mkdir(parents=True, exist_ok=True)
    (out_dir/"predictions").mkdir(parents=True, exist_ok=True)
    (out_dir/"reports").mkdir(parents=True, exist_ok=True)
    assert train_csv.exists() and val_csv.exists() and test_csv.exists()

    train_ds = ImgCSV(str(train_csv), make_tfms(True))
    val_ds   = ImgCSV(str(val_csv),   make_tfms(False))
    test_ds  = ImgCSV(str(test_csv),  make_tfms(False))

    y_tr = np.array([y for _,y in train_ds])
    cnts = np.array([(y_tr==0).sum(), (y_tr==1).sum()], dtype=np.int64)
    class_w = (cnts.sum()/(2.0*np.maximum(cnts,1))).astype(np.float32)
    sample_w = np.where(y_tr==0, class_w[0], class_w[1]).astype(np.float32)
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

    vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    vit.heads = nn.Linear(vit.heads.head.in_features, 2)
    n = inject_mlp_lora(vit, r=8, alpha=8, dropout=0.05)
    vit.to(device)

    crit = nn.CrossEntropyLoss(weight=torch.tensor(class_w, device=device))
    # train only LoRA + head
    params = [p for n,p in vit.named_parameters() if p.requires_grad]
    opt = AdamW(params, lr=2e-4, weight_decay=1e-4)

    best_auc, best_ep = -1, 0
    BEST = (out_dir/"checkpoints"/"vit_lora_best.pt")

    def run_epoch(loader, train=True):
        vit.train(train)
        losses, ys, ps = [], [], []
        for xb,yb in loader:
            xb,yb = xb.to(device), yb.to(device)
            with torch.set_grad_enabled(train):
                logits = vit(xb)
                loss = crit(logits, yb)
                if train:
                    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            probs = F.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            losses.append(loss.item()); ys.append(yb.cpu().numpy()); ps.append(probs)
        y = np.concatenate(ys); p = np.concatenate(ps)
        auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
        pr,rc,f1,_ = precision_recall_fscore_support(y,(p>=0.5).astype(int),average='binary',zero_division=0)
        return np.mean(losses), f1, auc, y, p

    patience, waited = 3, 0
    for ep in range(1, 20+1):
        tr = run_epoch(train_loader, True)
        vl = run_epoch(val_loader, False)
        print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[1]:.3f} auc {tr[2]:.3f} | val loss {vl[0]:.4f} f1 {vl[1]:.3f} auc {vl[2]:.3f}")
        if (vl[2] if not math.isnan(vl[2]) else -1) > best_auc:
            best_auc, best_ep, waited = vl[2], ep, 0
            torch.save(vit.state_dict(), BEST)
        else:
            waited += 1
            if waited>=patience:
                print(f"Early stop at {ep} (best AUC@{best_ep}={best_auc:.3f})"); break

    vit.load_state_dict(torch.load(BEST, map_location=device)); vit.eval()
    vl = run_epoch(val_loader, False); ts = run_epoch(test_loader, False)

    tau, policy = choose_tau(vl[4], vl[3], policy_prec=0.98)
    (out_dir/"reports"/"operating_threshold.txt").write_text(f"{tau:.6f}")

    def at_tau(name, y, p, t):
        yhat = (p>=t).astype(int)
        pr,rc,f1,_ = precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
        cm  = confusion_matrix(y,yhat,labels=[0,1]).tolist()
        acc = (yhat==y).mean()
        print(f"[{name}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f} | τ*={t:.6f} ({policy})")
        return dict(acc=float(acc),prec=float(pr),rec=float(rc),f1=float(f1),auc=float(auc),cm=cm,tau=float(t))

    val_summary  = at_tau("VAL @τ*",  vl[3], vl[4], tau)
    test_summary = at_tau("TEST @τ*", ts[3], ts[4], tau)

    # save preds
    def rows_from_ds(ds, probs, split, key="prob_vit"):
        return [dict(filepath=ds.fp[i], y_true=int(ds.y[i]), **{key: float(probs[i])}, split=split) for i in range(len(ds))]
    pred_csv = (out_dir/"predictions"/"vit_lora_val_test_predictions.csv")
    pd.DataFrame(rows_from_ds(val_ds,vl[4],"val")+rows_from_ds(test_ds,ts[4],"test")).to_csv(pred_csv, index=False)

    manifest = {
        "model":"ViT-B16-MLP-LoRA",
        "tau_star": float(tau),
        "paths":{"checkpoint":str(BEST),"predictions_csv":str(pred_csv),"tau_file":str(out_dir/'reports'/'operating_threshold.txt')},
        "val_at_tau": val_summary,"test_at_tau": test_summary
    }
    (out_dir/"reports"/"manifest_vit_lora.json").write_text(json.dumps(manifest, indent=2))
    return str(pred_csv), float(tau)

folds = [p for p in Path(BASE_LOSO).iterdir() if p.is_dir()]
print("Folds:", [f.name for f in folds])
for f in folds:
    print("\n=== ViT-LoRA:", f.name, "===")
    _ = train_one_fold(str(f))


Folds: ['Cough_Audio_Coswera', 'Cough_Audio_COUGHVID']

=== ViT-LoRA: Cough_Audio_Coswera ===
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:06<00:00, 54.4MB/s] 


Epoch 01 | train loss 0.0793 f1 0.829 auc 0.966 | val loss 0.0257 f1 0.800 auc 1.000
Epoch 02 | train loss 0.0032 f1 0.992 auc 1.000 | val loss 0.0009 f1 1.000 auc 1.000
Epoch 03 | train loss 0.0506 f1 0.936 auc 0.967 | val loss 0.0875 f1 0.667 auc 0.988
Epoch 04 | train loss 0.0067 f1 0.990 auc 0.993 | val loss 0.0546 f1 0.727 auc 0.994
Early stop at 4 (best AUC@1=1.000)
[VAL @τ*] acc=0.965 prec=1.000 rec=0.250 f1=0.400 auc=1.000 | τ*=0.999810 (prec>=0.98)
[TEST @τ*] acc=0.993 prec=1.000 rec=0.333 f1=0.500 auc=1.000 | τ*=0.999810 (prec>=0.98)

=== ViT-LoRA: Cough_Audio_COUGHVID ===
Epoch 01 | train loss 0.0132 f1 0.943 auc 0.995 | val loss 0.0214 f1 0.750 auc 0.997
Epoch 02 | train loss 0.0009 f1 0.995 auc 0.998 | val loss 0.0275 f1 0.750 auc 0.998
Epoch 03 | train loss 0.0007 f1 0.996 auc 0.999 | val loss 0.0327 f1 0.667 auc 0.999
Epoch 04 | train loss 0.0004 f1 0.999 auc 1.000 | val loss 0.0239 f1 0.750 auc 0.999
Epoch 05 | train loss 0.0005 f1 0.998 auc 0.999 | val loss 0.0162 f1 0

In [13]:
# === LOSO ENSEMBLE: merge per-fold predictions and evaluate ===
import os, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

BASE_LOSO = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
META = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/master_metadata_realpaths.csv"

def load_fold_preds(fold_dir):
    fold_dir = Path(fold_dir)
    paths = {
        "effnet": fold_dir/"effnet"/"predictions_val_test.csv",  # your effnet file name might be 'predictions_val_test.csv' or 'predictions_val_test.csv'
        "resnet": fold_dir/"resnet18"/"predictions"/"resnet18_val_test_predictions.csv",
        "vit":    fold_dir/"vit_lora"/"predictions"/"vit_lora_val_test_predictions.csv",
    }
    dfs = {}
    for k,p in paths.items():
        if p.exists():
            df = pd.read_csv(p)
            # normalize column names
            if k=="effnet":
                # detect column:
                prob_col = next((c for c in df.columns if c.startswith("prob")), None)
                df = df.rename(columns={prob_col: "prob_effnet"})
            elif k=="resnet":
                df = df.rename(columns={"prob_resnet":"prob_resnet"})
            else:
                df = df.rename(columns={"prob_vit":"prob_vit"})
            dfs[k]=df[["filepath","y_true","split", f"prob_{k}"]]
    return dfs

def pick_tau(val_probs, val_y, target_prec=0.98):
    taus = np.unique(np.clip(np.r_[0.0, val_probs, 1.0], 0,1))
    # best F1
    best_f1, best_tau = -1, 0.5
    for t in taus:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y,yhat,average='binary',zero_division=0)
        if f1>best_f1: best_f1, best_tau = f1, float(t)
    # precision target
    for t in taus[::-1]:
        yhat = (val_probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y,yhat,average='binary',zero_division=0)
        if pr>=target_prec: return float(t), "prec>=%.2f"%target_prec
    return best_tau, "maxF1"

def at_tau(y,p,t,name):
    yhat = (p>=t).astype(int)
    pr,rc,f1,_ = precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
    auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
    cm  = confusion_matrix(y,yhat,labels=[0,1]).tolist()
    acc = (yhat==y).mean()
    print(f"[{name}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return dict(acc=float(acc),prec=float(pr),rec=float(rc),f1=float(f1),auc=float(auc),cm=cm)

meta = pd.read_csv(META)[["filepath","source","label"]]
folds = [p for p in Path(BASE_LOSO).iterdir() if p.is_dir()]

pooled = []
for f in folds:
    print("\n==========================")
    print("FOLD:", f.name)
    dfs = load_fold_preds(f)
    assert "effnet" in dfs, f"EffNet preds missing in {f}"
    # progressive inner-join so we only ensemble rows present in all models available
    ens = dfs["effnet"]
    for k in ["resnet","vit"]:
        if k in dfs:
            ens = ens.merge(dfs[k], on=["filepath","y_true","split"], how="inner")

    # compute mean prob across available models
    prob_cols = [c for c in ens.columns if c.startswith("prob_")]
    ens["prob_ens"] = ens[prob_cols].mean(axis=1)

    # merge metadata for source/label filters
    ens = ens.merge(meta, on="filepath", how="left")

    # τ* from VAL
    val = ens[ens.split=="val"].copy()
    test= ens[ens.split=="test"].copy()
    tau, policy = pick_tau(val["prob_ens"].values, val["y_true"].values, target_prec=0.98)
    print("Chosen τ* =", f"{tau:.6f}", "| policy:", policy)

    print("[VAL @ τ*]")
    _ = at_tau(val["y_true"].values, val["prob_ens"].values, tau, "VAL")

    print("[TEST @ τ*]")
    mtest = at_tau(test["y_true"].values, test["prob_ens"].values, tau, "TEST")

    # Kaggle-only / no-Kaggle breakdowns (optional)
    for split in ["val","test"]:
        d = ens[ens.split==split].copy()
        kag = d[d["source"].isin(["kaggle_cancer_raw","kaggle_normal_raw"])].copy()
        nok = d[~d["source"].isin(["kaggle_cancer_raw","kaggle_normal_raw"])].copy()
        print(f"  {split.upper()} Kaggle-only:")
        if len(kag):
            _ = at_tau((kag.label=="cancer").astype(int).values, kag["prob_ens"].values, tau, f"{split} Kaggle-only")
        else:
            print("    (none)")
        if len(nok):
            _ = at_tau((nok.label=="cancer").astype(int).values,  nok["prob_ens"].values,  tau, f"{split} no-Kaggle")
        else:
            print("    (none)")

    pooled.append(test[["filepath","y_true","prob_ens"]])

# pooled metrics across folds (per-fold τ* is not re-applied here; this is simple pooling of probs)
if pooled:
    pool = pd.concat(pooled, ignore_index=True)
    # Optional: use 0.5 or find global τ on pooled VALs (not included here).
    # Just show AUC and best-F1 for pooled test:
    from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
    y = pool["y_true"].values; p = pool["prob_ens"].values
    tau_f1, _ = pick_tau(p, y, target_prec=0.0)
    yhat = (p>=tau_f1).astype(int)
    pr,rc,f1,_ = precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
    auc = roc_auc_score(y,p) if len(np.unique(y))>1 else float('nan')
    cm  = confusion_matrix(y,yhat,labels=[0,1]).tolist()
    print("\n=== POOLED TEST (using best-F1 on pooled test for display only) ===")
    print(f"acc={(yhat==y).mean():.3f}  prec={pr:.3f}  rec={rc:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print("CM:", cm)



FOLD: Cough_Audio_Coswera
Chosen τ* = 0.999850 | policy: prec>=0.98
[VAL @ τ*]
[VAL] acc=0.965 prec=1.000 rec=0.250 f1=0.400 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[81, 0], [3, 1]]
[TEST @ τ*]
[TEST] acc=0.993 prec=1.000 rec=0.333 f1=0.500 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[281, 0], [2, 1]]
  VAL Kaggle-only:
[val Kaggle-only] acc=0.571 prec=1.000 rec=0.250 f1=0.400 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[3, 0], [3, 1]]
[val no-Kaggle] acc=1.000 prec=0.000 rec=0.000 f1=0.000 auc=nan
CM (rows=true [healthy,cancer], cols=pred):
 [[78, 0], [0, 0]]
  TEST Kaggle-only:
[test Kaggle-only] acc=0.333 prec=1.000 rec=0.333 f1=0.500 auc=nan
CM (rows=true [healthy,cancer], cols=pred):
 [[0, 0], [2, 1]]
[test no-Kaggle] acc=1.000 prec=0.000 rec=0.000 f1=0.000 auc=nan
CM (rows=true [healthy,cancer], cols=pred):
 [[281, 0], [0, 0]]

FOLD: Cough_Audio_COUGHVID
Chosen τ* = 0.999980 | policy: prec>=0.98
[VAL @ τ*]
[VAL] acc=0.991 prec=1.000 rec=0.

In [16]:
# === ONE-CELL ENSEMBLE WITH ROBUST τ* (KAGGLE-ONLY VAL) ===
import os, json, math, pathlib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix
)

# ---------------------
# Paths (adjust only if you moved things)
# ---------------------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning")

PRED_PATHS = {
    "resnet":   ROOT / "Project_PreProess_harmonize/Project/Final_Master/predictions/resnet18_val_test_predictions.csv",
    "vit":      ROOT / "ViT-Model/predictions/vit_lora_val_test_predictions.csv",
    "effnet":   ROOT / "EffNet-Model/predictions/efficientnet_b0_val_test_predictions.csv",
}

META_CSV   = ROOT / "EffNet-Model/EffNet_Model_code/metadata/master_metadata_realpaths.csv"

OUT_DIR    = ROOT / "Ensemble"
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)

# ---------------------
# Helpers
# ---------------------
def load_preds(csv_path: Path, tag: str):
    if not csv_path.exists():
        print(f"⚠️  Missing predictions for {tag}: {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    # expected columns include: filepath, split, y_true, prob_cancer, (maybe label/source too)
    # normalize columns
    if "prob_cancer" not in df.columns:
        # try common fallbacks
        if "prob" in df.columns: df = df.rename(columns={"prob": "prob_cancer"})
        elif "p_cancer" in df.columns: df = df.rename(columns={"p_cancer": "prob_cancer"})
        else:
            raise ValueError(f"{tag} file lacks 'prob_cancer' column: {csv_path}")
    need = {"filepath","split","y_true","prob_cancer"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{tag} file missing columns {missing}: {csv_path}")
    # keep only needed + pass-through label/source if present
    keep = [c for c in ["filepath","split","y_true","prob_cancer","label","source","strata"] if c in df.columns]
    df = df[keep].copy()
    df = df.rename(columns={"prob_cancer": f"prob_{tag}"})
    return df

def safe_merge(dfs, on=("filepath","split")):
    out = None
    for d in dfs:
        if d is None: continue
        out = d if out is None else out.merge(d, on=list(on), how="outer")
    if out is None:
        raise RuntimeError("No prediction CSVs were found. Train/save model predictions first.")
    return out

def attach_source(df, meta_csv):
    if "source" in df.columns and df["source"].notna().any():
        return df  # already there
    meta = pd.read_csv(meta_csv)[["filepath","source"]]
    return df.merge(meta, on="filepath", how="left")

def metrics_at_tau(df, prob_col, tau, title=""):
    y = df["y_true"].astype(int).values
    p = df[prob_col].astype(float).values
    yhat = (p >= float(tau)).astype(int)
    acc  = accuracy_score(y, yhat)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    # AUC needs both classes present
    try:
        auc = roc_auc_score(y, p) if (y.min()!=y.max()) else float("nan")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    if title:
        print(f"[{title}] acc={acc:.3f}  prec={pr:.3f}  rec={rc:.3f}  f1={f1:.3f}  auc={auc:.3f}")
        print("CM (rows=true [healthy,cancer], cols=pred):")
        print(np.array(cm))
    return {"acc":acc,"prec":pr,"rec":rc,"f1":f1,"auc":auc,"cm":cm}

def choose_tau_robust(val_probs, val_y, *, target_prec=0.98, min_recall=0.50, min_tp=2, cap=0.99):
    """
    Precision-target if it doesn't destroy recall; else best-F1.
    Caps tau to avoid ~1.0 locks on tiny VAL (e.g., 6 positives).
    """
    val_probs = np.asarray(val_probs, dtype=float)
    val_y     = np.asarray(val_y, dtype=int)
    taus = np.unique(np.clip(np.r_[0.0, val_probs, 1.0], 0, cap))
    # Best-F1 (fallback)
    best_f1, best_tau = -1.0, 0.5
    for t in taus:
        yhat = (val_probs >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = f1, float(t)
    # Precision-target with safety floors
    for t in taus[::-1]:
        yhat = (val_probs >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average="binary", zero_division=0)
        tp = int(((yhat == 1) & (val_y == 1)).sum())
        if pr >= target_prec and rc >= min_recall and tp >= min_tp:
            return float(t), f"prec>={target_prec:.2f} (rc≥{min_recall:.2f}, tp≥{min_tp})"
    return float(best_tau), "maxF1 (robust fallback)"

def print_split_breakdowns(df, tau, prob_col, split_name):
    print(f"\n==== {split_name.upper()} ====  (τ* = {tau:.6f} )")
    # Kaggle-only
    kag_mask = df["source"].isin({"kaggle_cancer_raw","kaggle_normal_raw"})
    kag = df[kag_mask]
    no_kag = df[~kag_mask]
    if len(kag):
        metrics_at_tau(kag, prob_col, tau, f"{split_name} Kaggle-only")
        if "label" in kag.columns:
            print("  breakdown:\n", kag["label"].value_counts())
    else:
        print(f"[{split_name} Kaggle-only] no rows")
    if len(no_kag):
        m = metrics_at_tau(no_kag, prob_col, tau, f"{split_name} no-Kaggle")
        src_counts = no_kag["source"].value_counts().to_dict()
        print("  sources:", src_counts)
    else:
        print(f"[{split_name} no-Kaggle] no rows")

# ---------------------
# 1) Load available predictions & build ensemble
# ---------------------
dfs = [load_preds(p, tag) for tag,p in PRED_PATHS.items()]
ens = safe_merge(dfs, on=("filepath","split"))

# keep a single y_true column (resolve any duplicates by prioritizing first non-null)
if "y_true_x" in ens.columns or "y_true_y" in ens.columns:
    ycols = [c for c in ens.columns if c.startswith("y_true")]
    ens["y_true"] = ens[ycols].bfill(axis=1).iloc[:,0].astype(int)
    ens = ens.drop(columns=[c for c in ycols if c!="y_true"])
else:
    ens["y_true"] = ens["y_true"].astype(int)

# attach source (for Kaggle/no-Kaggle filtering)
ens = attach_source(ens, META_CSV)

# Which prob columns exist?
prob_cols = [c for c in ("prob_resnet","prob_vit","prob_effnet") if c in ens.columns]
if not prob_cols:
    raise RuntimeError("No prob_* columns found. Check that at least one model's CSV exists and has prob_cancer.")

# weights (equal by default)
weights = np.array([1.0]*len(prob_cols), dtype=float)
ens["prob_ens"] = np.average(ens[prob_cols].values, axis=1, weights=weights)

# ---------------------
# 2) Choose robust τ* on Kaggle-only VAL (fallback to full VAL if too small)
# ---------------------
val = ens[ens["split"]=="val"].copy()
test = ens[ens["split"]=="test"].copy()

val_kag = val[val["source"].isin({"kaggle_cancer_raw","kaggle_normal_raw"})]
if len(val_kag) >= 4:
    vy, vp = val_kag["y_true"].values, val_kag["prob_ens"].values
else:
    vy, vp = val["y_true"].values, val["prob_ens"].values

tau, policy = choose_tau_robust(vp, vy, target_prec=0.98, min_recall=0.50, min_tp=2, cap=0.99)
print(f"Chosen τ*={tau:.6f}  policy={policy}")

# ---------------------
# 3) Evaluate & print
# ---------------------
val_metrics  = metrics_at_tau(val,  "prob_ens", tau, "VAL @ τ*")
test_metrics = metrics_at_tau(test, "prob_ens", tau, "TEST @ τ*")
print_split_breakdowns(val,  tau, "prob_ens", "val")
print_split_breakdowns(test, tau, "prob_ens", "test")

# ---------------------
# 4) Save predictions, threshold, manifest
# ---------------------
pred_out = ens.copy()
pred_out["prob_tau_star"] = tau
pred_out["y_pred_tau_star"] = (pred_out["prob_ens"] >= tau).astype(int)
pred_path = OUT_DIR / "predictions/ensemble_val_test_predictions.csv"
pred_out.to_csv(pred_path, index=False)

(OUT_DIR / "reports" / "operating_threshold.txt").write_text(f"{tau:.6f}\n")
manifest = {
    "ensemble": {
        "members": [c.replace("prob_","") for c in prob_cols],
        "weights": weights.tolist(),
        "tau_star": float(tau),
        "tau_policy": policy,
        "paths": {
            "predictions_csv": str(pred_path),
            "tau_txt": str(OUT_DIR / "reports" / "operating_threshold.txt"),
        },
        "val_metrics_at_tau": val_metrics,
        "test_metrics_at_tau": test_metrics,
    }
}
with open(OUT_DIR / "reports" / "manifest_ensemble.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n✅ Saved:")
print(" -", pred_path)
print(" -", OUT_DIR / "reports" / "operating_threshold.txt")
print(" -", OUT_DIR / "reports" / "manifest_ensemble.json")


⚠️  Missing predictions for resnet: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/predictions/resnet18_val_test_predictions.csv
Chosen τ*=0.990000  policy=maxF1 (robust fallback)
[VAL @ τ*] acc=0.999  prec=0.857  rec=1.000  f1=0.923  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[733   1]
 [  0   6]]
[TEST @ τ*] acc=0.997  prec=0.750  rec=1.000  f1=0.857  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[366   1]
 [  0   3]]

==== VAL ====  (τ* = 0.990000 )
[val Kaggle-only] acc=0.909  prec=0.857  rec=1.000  f1=0.923  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[4 1]
 [0 6]]
[val no-Kaggle] acc=1.000  prec=0.000  rec=0.000  f1=0.000  auc=nan
CM (rows=true [healthy,cancer], cols=pred):
[[729   0]
 [  0   0]]
  sources: {'Cough_Audio_Coswera': 562, 'Cough_Audio_COUGHVID': 167}

==== TEST ====  (τ* = 0.990000 )
[test Kaggle-only] acc=0.800  prec=0.750  rec=1.000  f1=0.857  auc=1.00

In [18]:
# === ONE-CELL ENSEMBLE (auto-discovery, robust τ*, diagnostics) ===
import os, json, math, glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# --------------------------
# Config (tweak here)
# --------------------------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning")
META_CSV   = ROOT / "Project_PreProess_harmonize/Project/Final_Master/metadata/master_metadata_realpaths.csv"
OUT_DIR    = ROOT / "Ensemble"
AGG        = "mean"      # one of: "mean" | "gmean" | "min" | "max"
TARGET_PREC = 0.98
MIN_RECALL  = 0.50
MIN_TP      = 2
TAU_CAP     = 0.99999    # was 0.99; raise for tiny Kaggle-VAL
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)

# Preferred locations (if missing, we auto-search)
PREF = {
    "resnet": str(ROOT / "Project_PreProess_harmonize/Project/Model_training/Resnet-18 Model/predictions/resnet18_val_test_predictions.csv"),
    "vit":    str(ROOT / "ViT-Model/predictions/vit_lora_val_test_predictions.csv"),
    "effnet": str(ROOT / "EffNet-Model/predictions/efficientnet_b0_val_test_predictions.csv"),
}

# --------------------------
# Helper functions
# --------------------------
def auto_find(tag, pref):
    """Return best-guess path for a model's prediction CSV."""
    if pref and os.path.exists(pref):
        return pref
    patterns = {
        "resnet": "**/resnet18*_val_test*_predictions*.csv",
        "vit":    "**/vit*_val_test*_predictions*.csv",
        "effnet": "**/efficientnet*_val_test*_predictions*.csv",
    }
    hits = glob.glob(str(ROOT / patterns[tag]), recursive=True)
    return sorted(hits, key=len)[0] if hits else None

def load_preds(csv_path: str, tag: str):
    if not csv_path or not os.path.exists(csv_path):
        print(f"⚠️  Missing predictions for {tag}: {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    # normalize
    if "prob_cancer" not in df.columns:
        if "prob" in df.columns: df = df.rename(columns={"prob": "prob_cancer"})
        elif "p_cancer" in df.columns: df = df.rename(columns={"p_cancer": "prob_cancer"})
        else:
            raise ValueError(f"{tag} file lacks 'prob_cancer' column: {csv_path}")
    need = {"filepath","split","y_true","prob_cancer"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"{tag} file missing columns {miss}: {csv_path}")
    keep = [c for c in ["filepath","split","y_true","prob_cancer","label","source","strata"] if c in df.columns]
    df = df[keep].copy()
    df = df.rename(columns={"prob_cancer": f"prob_{tag}"})
    return df

def safe_merge(dfs, on=("filepath","split")):
    out = None
    for d in dfs:
        if d is None: continue
        out = d if out is None else out.merge(d, on=list(on), how="outer")
    if out is None:
        raise RuntimeError("No prediction CSVs were found. Train/save model predictions first.")
    return out

def attach_source(df, meta_csv):
    if "source" in df.columns and df["source"].notna().any():
        return df
    meta = pd.read_csv(meta_csv)[["filepath","source"]]
    return df.merge(meta, on="filepath", how="left")

def metrics_at_tau(df, prob_col, tau, title=""):
    y = df["y_true"].astype(int).values
    p = df[prob_col].astype(float).values
    yhat = (p >= float(tau)).astype(int)
    acc  = accuracy_score(y, yhat)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y, p) if (y.min()!=y.max()) else float("nan")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    if title:
        print(f"[{title}] acc={acc:.3f}  prec={pr:.3f}  rec={rc:.3f}  f1={f1:.3f}  auc={auc:.3f}")
        print("CM (rows=true [healthy,cancer], cols=pred):")
        print(np.array(cm))
    return {"acc":acc,"prec":pr,"rec":rc,"f1":f1,"auc":auc,"cm":cm}

def choose_tau_robust(val_probs, val_y, *, target_prec=0.98, min_recall=0.50, min_tp=2, cap=0.99999):
    val_probs = np.asarray(val_probs, dtype=float)
    val_y     = np.asarray(val_y, dtype=int)
    taus = np.unique(np.clip(np.r_[0.0, val_probs, 1.0], 0, cap))
    # Best-F1 fallback
    best_f1, best_tau = -1.0, 0.5
    for t in taus:
        yhat = (val_probs >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = f1, float(t)
    # Precision-target with safety floors
    for t in taus[::-1]:
        yhat = (val_probs >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(val_y, yhat, average="binary", zero_division=0)
        tp = int(((yhat == 1) & (val_y == 1)).sum())
        if pr >= target_prec and rc >= min_recall and tp >= min_tp:
            return float(t), f"prec>={target_prec:.2f} (rc≥{min_recall:.2f}, tp≥{min_tp})"
    return float(best_tau), "maxF1 (robust fallback)"

def print_split_breakdowns(df, tau, prob_col, split_name):
    print(f"\n==== {split_name.upper()} ====  (τ* = {tau:.6f} )")
    kag_mask = df["source"].isin({"kaggle_cancer_raw","kaggle_normal_raw"})
    kag = df[kag_mask]; no_kag = df[~kag_mask]
    if len(kag):
        metrics_at_tau(kag, prob_col, tau, f"{split_name} Kaggle-only")
        if "label" in kag.columns:
            print("  breakdown:\n", kag["label"].value_counts())
    else:
        print(f"[{split_name} Kaggle-only] no rows")
    if len(no_kag):
        _ = metrics_at_tau(no_kag, prob_col, tau, f"{split_name} no-Kaggle")
        print("  sources:", no_kag["source"].value_counts().to_dict())
    else:
        print(f"[{split_name} no-Kaggle] no rows")

def agg_probs(frame, cols, how):
    X = frame[cols].values
    if how == "mean":
        return np.mean(X, axis=1)
    if how == "gmean":
        Xc = np.clip(X, 1e-9, 1-1e-9)
        return np.exp(np.mean(np.log(Xc), axis=1))
    if how == "min":
        return np.min(X, axis=1)
    if how == "max":
        return np.max(X, axis=1)
    raise ValueError(f"Unknown AGG='{how}'")

# --------------------------
# 1) Load predictions (auto-discovery)
# --------------------------
paths = {k: (PREF.get(k) or None) for k in ["resnet","vit","effnet"]}
for k in list(paths.keys()):
    if not paths[k] or not os.path.exists(paths[k]):
        alt = auto_find(k, paths[k])
        if alt: paths[k] = alt
for k,p in paths.items():
    print(f"{k:>7}: {p if p and os.path.exists(p) else '— not found —'}")

dfs = [load_preds(paths[k], k) for k in ["resnet","vit","effnet"]]
ens = safe_merge(dfs, on=("filepath","split"))

# unify y_true
if "y_true_x" in ens.columns or "y_true_y" in ens.columns:
    ycols = [c for c in ens.columns if c.startswith("y_true")]
    ens["y_true"] = ens[ycols].bfill(axis=1).iloc[:,0].astype(int)
    ens = ens.drop(columns=[c for c in ycols if c!="y_true"])
else:
    ens["y_true"] = ens["y_true"].astype(int)

# attach source
ens = attach_source(ens, META_CSV)

# ensemble prob
prob_cols = [c for c in ("prob_resnet","prob_vit","prob_effnet") if c in ens.columns]
if not prob_cols:
    raise RuntimeError("No prob_* columns found. Check at least one model CSV.")
ens["prob_ens"] = agg_probs(ens, prob_cols, AGG)

# --------------------------
# 2) Choose τ* on Kaggle-VAL (fallback to full VAL)
# --------------------------
val  = ens[ens["split"]=="val"].copy()
test = ens[ens["split"]=="test"].copy()

val_kag = val[val["source"].isin({"kaggle_cancer_raw","kaggle_normal_raw"})]
if len(val_kag) >= 4:
    vy, vp = val_kag["y_true"].values, val_kag["prob_ens"].values
else:
    vy, vp = val["y_true"].values, val["prob_ens"].values

tau, policy = choose_tau_robust(vp, vy, target_prec=TARGET_PREC, min_recall=MIN_RECALL, min_tp=MIN_TP, cap=TAU_CAP)
print(f"\nChosen τ*={tau:.6f}  policy={policy} | AGG={AGG}")

# --------------------------
# 3) Evaluate & print
# --------------------------
val_metrics  = metrics_at_tau(val,  "prob_ens", tau, "VAL @ τ*")
test_metrics = metrics_at_tau(test, "prob_ens", tau, "TEST @ τ*")
print_split_breakdowns(val,  tau, "prob_ens", "val")
print_split_breakdowns(test, tau, "prob_ens", "test")

# Diagnostics: show healthy FPs with per-model probs (top 10 by prob_ens)
def fp_table(d, split_name):
    d = d.copy()
    d["y_pred"] = (d["prob_ens"] >= tau).astype(int)
    bad = d[(d["y_true"]==0) & (d["y_pred"]==1)].copy()
    if len(bad)==0:
        print(f"\nNo healthy FPs in {split_name}.")
        return bad
    cols = ["filepath","source","prob_ens"] + prob_cols
    bad = bad.sort_values("prob_ens", ascending=False)[cols].head(10)
    print(f"\nTop healthy FPs ({split_name}, up to 10):")
    print(bad.to_string(index=False))
    return bad

_ = fp_table(val, "VAL")
_ = fp_table(test, "TEST")

# --------------------------
# 4) Save predictions + τ* + manifest
# --------------------------
pred_out = ens.copy()
pred_out["prob_tau_star"] = tau
pred_out["y_pred_tau_star"] = (pred_out["prob_ens"] >= tau).astype(int)
pred_path = OUT_DIR / "predictions/ensemble_val_test_predictions.csv"
pred_out.to_csv(pred_path, index=False)

(OUT_DIR / "reports" / "operating_threshold.txt").write_text(f"{tau:.6f}\n")
manifest = {
    "ensemble": {
        "members": [c.replace("prob_","") for c in prob_cols],
        "aggregator": AGG,
        "tau_star": float(tau),
        "tau_policy": policy,
        "settings": {"TARGET_PREC":TARGET_PREC, "MIN_RECALL":MIN_RECALL, "MIN_TP":MIN_TP, "TAU_CAP":TAU_CAP},
        "paths": {
            "predictions_csv": str(pred_path),
            "tau_txt": str(OUT_DIR / "reports" / "operating_threshold.txt"),
        },
        "val_metrics_at_tau": val_metrics,
        "test_metrics_at_tau": test_metrics,
    }
}
with open(OUT_DIR / "reports" / "manifest_ensemble.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n✅ Saved:")
print(" -", pred_path)
print(" -", OUT_DIR / "reports" / "operating_threshold.txt")
print(" -", OUT_DIR / "reports" / "manifest_ensemble.json")


 resnet: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/Resnet-18 Model/predictions/resnet18_val_test_predictions.csv
    vit: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/ViT-Model/predictions/vit_lora_val_test_predictions.csv
 effnet: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/predictions/efficientnet_b0_val_test_predictions.csv

Chosen τ*=0.999902  policy=prec>=0.98 (rc≥0.50, tp≥2) | AGG=mean
[VAL @ τ*] acc=0.996  prec=1.000  rec=0.500  f1=0.667  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[734   0]
 [  3   3]]
[TEST @ τ*] acc=0.995  prec=1.000  rec=0.333  f1=0.500  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[367   0]
 [  2   1]]

==== VAL ====  (τ* = 0.999902 )
[val Kaggle-only] acc=0.727  prec=1.000  rec=0.500  f1=0.667  auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
[[5 0]
 [3 3]]
  b

In [19]:
# === Final Packaging Script (one cell) =======================================
import os, shutil, json, glob, textwrap
from pathlib import Path
import pandas as pd

# ---------- 1) Configure roots (edit if your layout changed) -----------------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning")

# Primary experiment roots
RESNET_ROOT   = ROOT / "Project_PreProess_harmonize/Project/Final_Master"
VIT_ROOT      = ROOT / "ViT-Model"
EFFNET_ROOT   = ROOT / "EffNet-Model"
ENSEMBLE_ROOT = ROOT / "Ensemble"

# Validation roots
LOSO_ROOT         = ROOT / "LOSO"
LOSO_BAL_ROOT     = ROOT / "LOSO_balanced"
KAGGLE_CV_ROOT_1  = ROOT / "Kaggle_CV"                 # original location
KAGGLE_CV_ROOT_2  = ROOT / "EffNet-Model/Kaggle_CV"    # later runs inside EffNet

# Output package
PKG = ROOT / "Final_Package"
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True, exist_ok=True)

LOG = []

def log(msg): 
    print(msg)
    LOG.append(msg)

def safe_copy(src: Path, dst: Path):
    """Copy file if exists; create parent; log results."""
    try:
        if src.exists() and src.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            log(f"✅ Copied: {src} -> {dst}")
        else:
            log(f"⚠️  Missing: {src}")
    except Exception as e:
        log(f"❌ Copy error: {src} -> {dst} | {e}")

def copy_glob(patterns, dst_dir: Path, keep_structure=False):
    """Copy many files by glob patterns (string or list of strings)."""
    if isinstance(patterns, (str, Path)):
        patterns = [patterns]
    matches = []
    for pat in patterns:
        matches.extend([Path(p) for p in glob.glob(str(pat), recursive=True)])
    if not matches:
        log(f"⚠️  No matches for patterns: {patterns}")
        return 0
    cnt=0
    for src in matches:
        if src.is_file():
            if keep_structure:
                rel = src.relative_to(src.anchor) if src.is_absolute() else src
                dst = dst_dir / rel
            else:
                dst = dst_dir / src.name
            safe_copy(src, dst)
            cnt+=1
    return cnt

# ---------- 2) Data & splits -------------------------------------------------
DATA_DIR = PKG / "data"
(DATA_DIR / "master/metadata").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "splits").mkdir(parents=True, exist_ok=True)

# master metadata (most up-to-date realpaths)
safe_copy(RESNET_ROOT / "metadata/master_metadata_realpaths.csv",
          DATA_DIR / "master/metadata/master_metadata_realpaths.csv")

# try to gather all split dirs we used across runs
possible_split_dirs = [
    RESNET_ROOT / "metadata/splits",
    VIT_ROOT     / "metadata/splits",
    EFFNET_ROOT  / "Eff_Net_Model_data/metadata/splits",
    ROOT / "Project_PreProess_harmonize/Project/Model_training/metadata/splits",
]
seen = set()
for d in possible_split_dirs:
    if d.exists():
        for f in ["train.csv", "val.csv", "test.csv"]:
            src = d / f
            if src.exists():
                # name them by parent dir to avoid overwrite
                tag = d.parent.name if d.parent else "splits"
                dst = DATA_DIR / f"splits/{tag}_{f}"
                safe_copy(src, dst)
                seen.add(str(d))
if not seen:
    log("⚠️  No base splits found in the usual locations.")

# ---------- 3) Figures (any PNGs we generated) ------------------------------
FIG_DIR = PKG / "report/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Common places we saved PNGs
figure_globs = [
    str(RESNET_ROOT / "reports/*.png"),
    str(VIT_ROOT / "reports/*.png"),
    str(EFFNET_ROOT / "reports/*.png"),
    str(ENSEMBLE_ROOT / "reports/*.png"),
    str(KAGGLE_CV_ROOT_1 / "aggregate*/**/*.png"),
    str(KAGGLE_CV_ROOT_1 / "**/*/reports/*.png"),
    str(KAGGLE_CV_ROOT_2 / "aggregate*/**/*.png"),
    str(KAGGLE_CV_ROOT_2 / "**/*/reports/*.png"),
]
copy_glob(figure_globs, FIG_DIR)

# ---------- 4) Models & predictions -----------------------------------------
MODELS_DIR = PKG / "models"
MODELS_DIR.mkdir(exist_ok=True)

def add_model_block(name, root, ckpt_rel, pred_rel, tau_rel, manifest_rel=None):
    mdir = MODELS_DIR / name
    mdir.mkdir(exist_ok=True, parents=True)
    # files
    safe_copy(root / ckpt_rel, mdir / Path(ckpt_rel).name)
    safe_copy(root / pred_rel, mdir / Path(pred_rel).name)
    safe_copy(root / tau_rel,  mdir / Path(tau_rel).name)
    if manifest_rel:
        safe_copy(root / manifest_rel, mdir / Path(manifest_rel).name)

# ResNet18 (if files still exist)
add_model_block(
    "resnet18",
    RESNET_ROOT,
    "checkpoints/resnet18_best.pt",
    "predictions/resnet18_val_test_predictions.csv",
    "reports/operating_threshold.txt",
    manifest_rel=None  # we may not have created one for resnet; skip
)

# ViT-LoRA
add_model_block(
    "vit_lora",
    VIT_ROOT,
    "checkpoints/vit_lora_best.pt",
    "predictions/vit_lora_val_test_predictions.csv",
    "reports/operating_threshold.txt",
    manifest_rel="manifest_vit.json"
)

# EfficientNet-B0
add_model_block(
    "efficientnet_b0",
    EFFNET_ROOT,
    "checkpoints/efficientnet_b0_best.pt",
    "predictions/efficientnet_b0_val_test_predictions.csv",
    "reports/operating_threshold.txt",
    manifest_rel="manifest_effnet.json"
)

# Ensemble (predictions + tau + manifest only)
ENS_DIR = MODELS_DIR / "ensemble"
ENS_DIR.mkdir(exist_ok=True, parents=True)
safe_copy(ENSEMBLE_ROOT / "predictions/ensemble_val_test_predictions.csv", ENS_DIR / "ensemble_val_test_predictions.csv")
safe_copy(ENSEMBLE_ROOT / "reports/operating_threshold.txt", ENS_DIR / "operating_threshold.txt")
safe_copy(ENSEMBLE_ROOT / "reports/manifest_ensemble.json",  ENS_DIR / "manifest_ensemble.json")

# ---------- 5) Validation: LOSO & Kaggle-CV ---------------------------------
VAL_DIR = PKG / "validation"
VAL_DIR.mkdir(exist_ok=True)

# (a) LOSO: copy all CSVs, thresholds, manifests, and splits folders if any
def copy_tree_filtered(src_root: Path, dst_root: Path, file_exts=(".csv", ".txt", ".json", ".png")):
    """Copy tree but only certain file types; preserve structure under dst_root/src_root.name/..."""
    if not src_root.exists():
        log(f"⚠️  LOSO source missing: {src_root}")
        return 0
    copied = 0
    for path in src_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in file_exts:
            rel = path.relative_to(src_root.parent)  # keep one level up to embed folder name
            dst = dst_root / rel
            safe_copy(path, dst)
            copied += 1
    return copied

copy_tree_filtered(LOSO_ROOT, VAL_DIR)
copy_tree_filtered(LOSO_BAL_ROOT, VAL_DIR)

# (b) Kaggle CV: copy per-fold runs + aggregates from both roots
for kroot in [KAGGLE_CV_ROOT_1, KAGGLE_CV_ROOT_2]:
    copy_tree_filtered(kroot, VAL_DIR)

# ---------- 6) Metrics summary (pull manifests if present) -------------------
summary = {"models": {}, "notes": []}

def maybe_load_json(p: Path):
    try:
        if p.exists():
            return json.loads(p.read_text())
    except Exception as e:
        log(f"⚠️  JSON read error: {p} | {e}")
    return None

# ViT manifest
vit_manifest = maybe_load_json(VIT_ROOT / "manifest_vit.json")
if vit_manifest:
    summary["models"]["vit_lora"] = vit_manifest

# EffNet manifest
eff_manifest = maybe_load_json(EFFNET_ROOT / "manifest_effnet.json")
if eff_manifest:
    summary["models"]["efficientnet_b0"] = eff_manifest

# Ensemble manifest
ens_manifest = maybe_load_json(ENSEMBLE_ROOT / "reports/manifest_ensemble.json")
if ens_manifest:
    summary["models"]["ensemble"] = ens_manifest

# ResNet: build a light manifest if predictions exist
res_pred = RESNET_ROOT / "predictions/resnet18_val_test_predictions.csv"
res_tau  = RESNET_ROOT / "reports/operating_threshold.txt"
if res_pred.exists():
    # lightweight summary (columns: filepath,label,source,strata,y_true,prob_cancer,y_pred_0p5,split)
    try:
        df = pd.read_csv(res_pred)
        val = df[df["split"]=="val"]
        test= df[df["split"]=="test"]
        # store only sizes; detailed metrics already printed in notebooks
        summary["models"]["resnet18"] = {
            "paths": {
                "checkpoint": str(RESNET_ROOT/"checkpoints/resnet18_best.pt") if (RESNET_ROOT/"checkpoints/resnet18_best.pt").exists() else None,
                "predictions_csv": str(res_pred),
                "operating_threshold_txt": str(res_tau) if res_tau.exists() else None,
            },
            "sizes": {"val": int(len(val)), "test": int(len(test))}
        }
    except Exception as e:
        log(f"⚠️  Could not read resnet predictions to summarise: {e}")
else:
    summary["models"]["resnet18"] = {
        "paths": {
            "checkpoint": str(RESNET_ROOT/"checkpoints/resnet18_best.pt") if (RESNET_ROOT/"checkpoints/resnet18_best.pt").exists() else None,
            "predictions_csv": None,
            "operating_threshold_txt": str(res_tau) if res_tau.exists() else None,
        },
        "sizes": {"val": None, "test": None}
    }

# Kaggle CV aggregate metrics if present (any *.json under validation/Kaggle_CV)
agg_jsons = list((VAL_DIR).rglob("aggregate*/**/*.json"))
agg_jsons += list((VAL_DIR).rglob("aggregate*/metrics*.json"))
kaggles = []
for j in agg_jsons:
    try:
        kaggles.append({"path": str(j), "metrics": json.loads(Path(j).read_text())})
    except Exception:
        pass
if kaggles:
    summary["kaggle_cv_aggregates"] = kaggles

# Save summary
(PKG / "report").mkdir(exist_ok=True, parents=True)
with open(PKG / "report/metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
log(f"✅ Wrote metrics_summary.json -> {PKG / 'report/metrics_summary.json'}")

# ---------- 7) REPORT.md (executive summary) --------------------------------
def md_h2(s): return f"\n## {s}\n"
def md_code(s): return "```\n" + s.strip() + "\n```\n"

report = []
report.append("# Final Package Report\n")
report.append(textwrap.dedent("""
**Project conclusion at a glance**

- The binary cough→cancer task is **confounded**: all *cancer* images come from **Kaggle**, and all *healthy* from **Coswara/COUGHVID**. 
- Multiple models achieved perfect scores on mixed-source splits, and a **source-only** classifier reproduces that performance ⇒ models learned **source cues**, not pathology.
- We therefore:
  1) Document leakage (LOSO, duplicate audit).
  2) Provide a fair, *separate* benchmark on **Kaggle-only** (5-fold CV) with calibrated thresholds.
"""))

# Data snapshot (if available)
meta_csv = DATA_DIR / "master/metadata/master_metadata_realpaths.csv"
if meta_csv.exists():
    try:
        mdf = pd.read_csv(meta_csv)
        by_label  = mdf["label"].value_counts().to_dict()
        by_source = mdf["source"].value_counts().to_dict()
        report.append(md_h2("Data snapshot"))
        report.append("- **Master rows**: {}".format(len(mdf)))
        report.append(f"- **Counts by label**: {by_label}")
        report.append(f"- **Counts by source**: {by_source}")
    except Exception as e:
        report.append(f"\n_Unable to summarise master metadata: {e}_\n")

# Model metrics
report.append(md_h2("Model artifacts & key metrics"))
for m in ["resnet18","vit_lora","efficientnet_b0","ensemble"]:
    report.append(f"### {m}")
    info = summary["models"].get(m, {})
    if not info:
        report.append("_No manifest found._\n")
        continue
    # show paths
    paths = info.get("paths", {})
    if paths:
        for k,v in paths.items():
            report.append(f"- {k}: `{v}`")
    # show metrics if present
    for k in ["metrics","tau_star","sizes"]:
        if k in info: 
            report.append(f"- {k}: `{info[k]}`")
    report.append("")

# Kaggle CV
if "kaggle_cv_aggregates" in summary and summary["kaggle_cv_aggregates"]:
    report.append(md_h2("Kaggle-only Cross-Validation (aggregate snapshots)"))
    for item in summary["kaggle_cv_aggregates"]:
        report.append(f"- `{item['path']}` → `{item['metrics']}`")

# Figures index
report.append(md_h2("Figures"))
pngs = sorted([p.name for p in FIG_DIR.glob("*.png")])
if pngs:
    report.append(f"_Copied {len(pngs)} figure(s). A selection:_")
    for p in pngs[:10]:
        report.append(f"- {p}")
else:
    report.append("_No figures found._")

# Limitations
report.append(md_h2("Limitations & next steps"))
report.append(textwrap.dedent("""
- **Confounding / leakage:** Model performance on mixed-source splits is not valid evidence of medical discrimination.
- **Fair evaluation path:** Use only datasets where both classes come from the same source/domain; we provide Kaggle-only CV as a small, valid benchmark.
- **Future work:** Acquire multi-source cancer data; consider domain-adversarial training or GroupDRO once the class-by-source entanglement is reduced; keep strict deduplication.
"""))

# Write REPORT.md
REPORT_MD = PKG / "report/REPORT.md"
REPORT_MD.parent.mkdir(parents=True, exist_ok=True)
REPORT_MD.write_text("\n".join(report))
log(f"✅ Wrote REPORT.md -> {REPORT_MD}")

# ---------- 8) Zip the package (optional) -----------------------------------
ZIP_PATH = str(PKG) + ".zip"
shutil.make_archive(str(PKG), 'zip', root_dir=str(PKG))
log(f"✅ Zipped package -> {ZIP_PATH}")

# ---------- 9) Final echo ----------------------------------------------------
print("\n" + "="*72)
print("Final package created at:")
print(f"  {PKG}")
print("Key files:")
print(f"  - {REPORT_MD}")
print(f"  - {PKG/'report/metrics_summary.json'}")
print(f"  - {ZIP_PATH}")
print("="*72)


⚠️  Missing: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/master_metadata_realpaths.csv
✅ Copied: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/splits/train.csv -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Final_Package/data/splits/metadata_train.csv
✅ Copied: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/splits/val.csv -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Final_Package/data/splits/metadata_val.csv
✅ Copied: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/splits/test.csv -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced M